# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb, os
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily_march': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"}
print("Connected.")

Connected.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — "What Predicts Health?" (ML Appendix, Random Forest
feature importance for Health Score)**

The paper reports Average Position (43%), Impressions (32%), and
Scroll Depth (15%) as the top predictors of Health Score.

*Where does the label come from?* Health Score is explicitly defined
earlier in the paper as a composite: Impressions (30 pts) + Position
(30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Three of the four
top-importance features (position, impressions, scroll depth) are
literally components the label is built from.

*Does the validation design carry the claim?* The paper is admirably
upfront about this - it states the target "is partly constructed from
some of these inputs, so importance is descriptive rather than
causal." I'd add one constructive point: an 80/20 holdout split, on
its own, doesn't fix this. When a label is a deterministic formula of
some features, no train/test split - however careful - prevents the
model from partially "solving the formula" rather than learning a
generalizable pattern, since the leak is structural (built into the
label's definition), not a train/test overlap issue. The paper's own
caveat is the right instinct; I'd only suggest naming this explicitly
as a limitation of the *label design*, not just the *interpretation*.

---

**Finding 2 — "What Predicts Growth?" (ML Appendix, logistic
regression, 71% holdout accuracy)**

The model predicts growing vs. declining pages using features like
Content Age, Days Since Update, Days Visible, and Average Position.

*Where does the label come from?* Trend direction, calculated from
30-day-vs-previous-30-day impression change (defined earlier in the
paper's "Understanding the Metrics" section) - structurally similar to
the label I built in my own ML-08/ML-09 work.

*Does the validation design carry the claim?* The methodology section
states an 80/20 split was used for the logistic regression, but
doesn't specify whether that split was random by row or grouped by
brand. Since the study spans 57 brands, and brands likely have
consistent internal practices (publishing cadence, refresh habits,
templates), a random row split could let pages from the same brand
appear in both train and test - letting the model partially learn
brand-specific quirks rather than a signal that generalizes across
brands. This is the same grouped-split lesson I applied to my own
model in ML-08 (client-holdout vs. random split) - I'd ask whether the
71% accuracy holds up under a brand-grouped holdout, the way I tested
mine.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No live query needed here - this section is a close reading of the
# paper's own reported methodology, not a re-analysis of its data.
# Documenting the two findings and questions as structured data for clarity:

findings_audit = {
    "Finding 1": {
        "claim": "Random forest feature importance for Health Score (Avg Position 43%, Impressions 32%, Scroll Depth 15%)",
        "label_source": "Health Score = Impressions(30) + Position(30) + CTR(20) + Scroll Depth(20) - a composite formula",
        "validation_question": "Does an 80/20 holdout split fix a label that is structurally derived from several of its own top features? (No - this is formula leakage, not split leakage.)"
    },
    "Finding 2": {
        "claim": "Logistic regression predicts growth vs decline at 71% holdout accuracy",
        "label_source": "Trend direction: 30-day vs previous-30-day impression change",
        "validation_question": "Was the 80/20 split random by row, or grouped by brand (57 brands in the dataset)? A random split risks brand-level leakage, the same issue I tested for in my own ML-08 client-holdout comparison."
    }
}

for name, details in findings_audit.items():
    print(f"\n{name}")
    for k, v in details.items():
        print(f"  {k}: {v}")


Finding 1
  claim: Random forest feature importance for Health Score (Avg Position 43%, Impressions 32%, Scroll Depth 15%)
  label_source: Health Score = Impressions(30) + Position(30) + CTR(20) + Scroll Depth(20) - a composite formula
  validation_question: Does an 80/20 holdout split fix a label that is structurally derived from several of its own top features? (No - this is formula leakage, not split leakage.)

Finding 2
  claim: Logistic regression predicts growth vs decline at 71% holdout accuracy
  label_source: Trend direction: 30-day vs previous-30-day impression change
  validation_question: Was the 80/20 split random by row, or grouped by brand (57 brands in the dataset)? A random split risks brand-level leakage, the same issue I tested for in my own ML-08 client-holdout comparison.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

This directly follows up on my methodology question for Finding 2 -
was the paper's 80/20 split random by row or grouped by brand? I can't
test that on their data, but I can test the equivalent question on my
own: does my model's score change meaningfully between a naive random
split and a client-grouped split? If it does, that gap is the amount
of "fake" performance a random split would have hidden.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np, pandas as pd

data = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            AVG(gsc_avg_position) AS avg_position,
            AVG(gsc_clicks) AS avg_clicks
        FROM {TABLES['fact_daily_march']}
        GROUP BY 1,2
        HAVING imp_first_half >= 20
    )
    SELECT *,
        CASE WHEN imp_second_half < 0.8 * imp_first_half THEN 1 ELSE 0 END AS is_declining
    FROM agg
""").df()

feature_cols = ['imp_first_half', 'avg_position', 'avg_clicks']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split (the risk I flagged in Finding 2)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    data[feature_cols].fillna(0), data['is_declining'], test_size=0.25, random_state=42, stratify=data['is_declining'])
model_r = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)
score_r = model_r.predict_proba(X_te_r)[:, 1]

# AFTER: grouped client-holdout split (same design used in ML-08)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
train, test = data.iloc[train_idx], data.iloc[test_idx]
X_tr_g, y_tr_g = train[feature_cols].fillna(0), train['is_declining']
X_te_g, y_te_g = test[feature_cols].fillna(0), test['is_declining']
model_g = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
score_g = model_g.predict_proba(X_te_g)[:, 1]

comparison = pd.DataFrame([
    {'split': 'random (before)', 'precision_at_20': round(precision_at_k(score_r, y_te_r.values, 20),3),
     'precision_at_50': round(precision_at_k(score_r, y_te_r.values, 50),3)},
    {'split': 'client-holdout (after)', 'precision_at_20': round(precision_at_k(score_g, y_te_g.values, 20),3),
     'precision_at_50': round(precision_at_k(score_g, y_te_g.values, 50),3)},
])
print(comparison)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                    split  precision_at_20  precision_at_50
0         random (before)             0.35             0.32
1  client-holdout (after)             0.35             0.40


**Result:** Precision@20 was identical between splits (0.35 vs 0.35).
At Precision@50, the client-holdout split actually scored *higher*
than the random split (0.40 vs 0.32) - the opposite of what I
expected when raising the brand-grouping concern for Finding 2.

**Honest interpretation:** This does not mean grouping doesn't matter
- it means that on this particular feature set (impressions, position,
clicks), the signal generalizes reasonably well across clients even
under a naive split, so client-level memorization wasn't a large
factor here. The random split's slightly lower Precision@50 is more
likely sample variance from a different train/test partition than
evidence of brand-level leakage inflating results.

This is a useful, honest counter-example to my own methodology
question about Finding 2: a grouped split is still the more defensible
default (it protects against a real risk even when that risk doesn't
show up every time), but this run shows the risk doesn't always
materialize as a large score inflation - it depends on how much
client-specific signal the features actually capture. My features here
are fairly generic (impressions, position, clicks), which may be why
client identity mattered less than it might for the paper's richer
feature set.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

This follows up on my methodology question for Finding 1 - can an
80/20 split alone fix a label that's structurally derived from its own
features? I already found this exact problem in my own ML-08 work
(imp_total leaking back into the label), so this section re-confirms
the fix held on my final feature set - the same "formula leakage"
issue I flagged in FlyRank's Health Score prediction.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Re-run the deliberate leak test on the FINAL feature set - same
# "formula leakage" pattern flagged in Finding 1 (Health Score built
# from its own top predictors)
feature_cols_leaky = feature_cols + ['imp_second_half']  # imp_second_half IS the label's source

X_leak = data[feature_cols_leaky].fillna(0)
y = data['is_declining']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xl_tr, yl_tr)
leaky_score = leaky_model.predict_proba(Xl_te)[:,1]
leaky_p50 = precision_at_k(leaky_score, yl_te.values, 50)

honest_p50 = precision_at_k(score_g, y_te_g.values, 50)

print(f"Honest feature set Precision@50: {honest_p50:.3f}")
print(f"With imp_second_half deliberately added back in: {leaky_p50:.3f}  <- should jump suspiciously high")
print(f"\nFinal feature set used: {feature_cols}")
print(f"Confirmed excluded: imp_second_half, imp_total, and any FlyRank product decision flags")

Honest feature set Precision@50: 0.400
With imp_second_half deliberately added back in: 1.000  <- should jump suspiciously high

Final feature set used: ['imp_first_half', 'avg_position', 'avg_clicks']
Confirmed excluded: imp_second_half, imp_total, and any FlyRank product decision flags


**Result:** Honest feature set (imp_first_half, avg_position,
avg_clicks) scores Precision@50 of 0.400. Deliberately adding
`imp_second_half` back in - the exact value the label is derived from
- pushes the score to a perfect 1.000. This is the unmistakable
signature of label leakage: no real model achieves perfect precision
on real-world data, so a jump to 1.000 confirms the feature is
reading the answer, not learning a pattern.

This confirms the fix from ML-08 held: my final feature set
(imp_first_half, avg_position, avg_clicks) does not contain
imp_second_half or imp_total, and produces an honest, non-perfect
score. It also directly validates my Finding 1 methodology question
about the paper's Health Score model - a formula-derived label can
make even simple features "leak,"

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence (from ML-08):** "The corrected result (0.35/0.42
model vs 0.00/0.06 baseline at k=20/50) is honest and shows a real,
defensible lift."

**Rewritten in safe language:** On this client-holdout test split of
March 2026 warehouse data, the random forest model achieved observed
Precision@20 of 0.35 and Precision@50 of 0.42, compared to a baseline
rule's 0.00 and 0.06. This is a directional signal that the model
finds patterns the baseline rule misses on this specific dataset and
split - it is decision-support for prioritizing a review queue, not a
claim that this lift holds on unseen data, other time periods, or in
production.

This mirrors the same discipline FlyRank's paper itself uses when it
writes that Random Forest importance "is descriptive rather than
causal" for Health Score, and that the logistic regression's 71%
accuracy should be "read as descriptive indicators... not as direct
instructions." I'm applying the same standard to my own numbers that I
asked of theirs.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Original claim: 'a real, defensible lift'")
print("Rewritten claim uses: observed, directional signal, decision-support")
print("Removed: any implication of guaranteed or universal performance")
print("\nConsistency check: same standard applied to my Finding 1/2 critique of the FlyRank paper")

Original claim: 'a real, defensible lift'
Rewritten claim uses: observed, directional signal, decision-support
Removed: any implication of guaranteed or universal performance

Consistency check: same standard applied to my Finding 1/2 critique of the FlyRank paper


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.